In [ ]:
from pathlib import Path
import os
import subprocess, sys, time
from datetime import datetime

import hashlib
import json
from pathlib import Path
import shutil
import torch

from huggingface_hub import snapshot_download
from omegaconf import OmegaConf
from safetensors.torch import load_file

REPO = Path(os.environ.get("NANOWM_REPO", Path.cwd())).resolve()
WORKSPACE = Path(os.environ.get("WM_ROLLOUTS_DIR", REPO.parent / "outputs/wm-rollouts"))
NUM_SAMPLES = 12
SAMPLING_STEPS = 50  # 15: quick preview; 50: default; 250: slower comparison, no quality guarantee
SEED = 42
BATCH_SIZE = 2  # keeps the memory footprint small
WORKSPACE.mkdir(parents=True, exist_ok=True)
ASSETS = WORKSPACE / "assets"

def run(argv, **kwargs):
    print("Running:", " ".join(map(str, argv)), flush=True)
    subprocess.run(list(map(str, argv)), check=True, **kwargs)

run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'CUDA GPU is not available in the current Python environment'; print(torch.cuda.get_device_name(0), torch.version.cuda)"])
print("Project:", REPO)
print("Workspace:", WORKSPACE)


## Choose checkpoint and validation data


In [ ]:
data_path = os.environ.get("WM_VALIDATION_DATA", str(REPO.parent / "outputs/wm_training_data/pushT_original/val"))

# MODEL_ID = "RNonnenmann/nanowm_b2_small_pusht"
# MODEL_REVISION = "13ba98942bac917d796433fc976410775389004e"
# MODEL_ID = "knightnemo/nanowm-b2-dino-wm-pusht-100k"
# MODEL_REVISION = "613919a559470a3b551160461627a3c1ac5cc17f"
ckpt = Path(os.environ.get("NANOWM_CHECKPOINT", REPO.parent / "outputs/trained-wm/latest"))

# Change action normalization stats!

VAE_ID = "stabilityai/sd-vae-ft-mse"
VAE_REVISION = "31f26fdeee1355a5c34592e401dd41e45d25a493"

OUTPUT = WORKSPACE / "pusht-finetuned-ER-original" / ckpt.name

### Load model and validation data

In [ ]:
def sha256(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

root = Path(ASSETS)
root.mkdir(parents=True, exist_ok=True)

# ckpt = Path(snapshot_download(MODEL_ID, revision=MODEL_REVISION,
#                             allow_patterns=["config.yaml", "model.safetensors"],
#                             local_dir=root / "checkpoint"))

# Download VAE
vae = snapshot_download(VAE_ID, revision=VAE_REVISION,
                        allow_patterns=["config.json", "diffusion_pytorch_model.safetensors"],
                        local_dir=root / "vae")
# The current production loader consumes a state dict via torch.load.
# Convert the trusted, revision-pinned safetensors once, atomically.
model_digest = sha256(ckpt / "model.safetensors")
pt = ckpt / "model.pt"
stamp = ckpt / "model.pt.source-sha256"
if not pt.exists() or not stamp.exists() or stamp.read_text().strip() != model_digest:
    temporary = ckpt / "model.pt.partial"
    torch.save(load_file(str(ckpt / "model.safetensors")), temporary)
    temporary.replace(pt)
    stamp.write_text(model_digest + "\n")

conf = OmegaConf.load(ckpt / "config.yaml")
# conf.dataset_dir = str(data.parent)
conf.vae_model_path = str(Path(vae).resolve())
# Training images are unused for inference; stats still use all train metadata.
conf.dataset.loader.data_path_val = data_path
conf.dataset.loader.split_ratio = 0.0
config_path = root / "rollout-config.yaml"
OmegaConf.save(conf, config_path)

## Generate predictions


In [ ]:
started = time.monotonic()
run([sys.executable,
     "src/sample/rollout_pusht.py",
     "--config", str(config_path), "--ckpt", str(pt),
     "--save_path", str(OUTPUT), "--num_samples", str(NUM_SAMPLES), "--batch_size", str(BATCH_SIZE),
     "--rollout_length", "20", "--history_length", "3",
     "--num_sampling_steps", str(SAMPLING_STEPS), "--scheduling_mode", "sequential",
     "--history_stabilization_level", "0.02", "--fps", "8",
     "--seed", str(SEED), "--unique_trajectories"], cwd=REPO)
report = json.loads((OUTPUT / "report.json").read_text())
assert len(report["samples"]) == NUM_SAMPLES
print(f"Finished in {time.monotonic()-started:.1f} seconds; peak model GPU allocation {report['peak_gpu_memory_gib']:.2f} GiB.")


## Watch every result

**Left = recorded validation images. Right = model prediction.** The first 3 frames are context.

In [ ]:
from IPython.display import display, Video, Markdown
for sample in report["samples"]:
    i = sample["sample_id"]
    display(Markdown(f"### Clip {i+1} · validation episode {sample['traj_idx']}\nRecorded (left) | predicted (right); 3 context frames + 17 predicted frames."))
    display(Video(filename=str(OUTPUT / f"sample_{i:04d}_compare.mp4"), embed=True, width=768, html_attributes='controls loop muted playsinline style="max-width:100%;height:auto"'))
rows = ["| Clip | Prediction PSNR / SSIM | Frozen last context PSNR / SSIM |", "|---|---|---|"]
for s in report["samples"]:
    a, b = s["prediction"], s["repeat_last_context"]
    rows.append(f"| {s['sample_id']+1} | {a['psnr_db']:.2f} dB / {a['ssim']:.4f} | {b['psnr_db']:.2f} dB / {b['ssim']:.4f} |")
display(Markdown("\n".join(rows)))

## Evaluate the checkpoint

In [ ]:
run([sys.executable, "src/sample/evaluate_metrics.py",
    "--video_dir", str(OUTPUT),
    "--history_length", "3" ,
    "--output_csv", OUTPUT / f"metrics.csv"], cwd=REPO)

## Visualize the metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Keep this in sync with --history_length used when generating/evaluating the rollout.
history_length = 3
summary_path = OUTPUT / "metrics_summary.csv"
metrics_summary = pd.read_csv(summary_path)

metric_names = ["mse", "psnr", "ssim", "lpips", "dreamsim"]
missing_metrics = set(metric_names) - set(metrics_summary.columns)
if missing_metrics:
    raise ValueError(f"Missing metrics in {summary_path}: {sorted(missing_metrics)}")

# frame_idx < history_length contains context frames; plot generated frames only.
rollout_metrics = metrics_summary.loc[
    metrics_summary["frame_idx"] >= history_length
].sort_values("frame_idx")
if rollout_metrics.empty:
    raise ValueError(f"No generated frames found in {summary_path}")

# Display the first generated frame as rollout step 1.
rollout_steps = rollout_metrics["frame_idx"] - history_length + 1
fig, axes = plt.subplots(
    len(metric_names), 1, figsize=(10, 12), sharex=True, constrained_layout=True
)
fig.suptitle("Metric evolution during rollout (context frames excluded)", fontsize=14)

for ax, metric in zip(axes, metric_names):
    values = rollout_metrics[metric]
    mean_value = values.mean()
    ax.plot(rollout_steps, values, marker="o", linewidth=1.8, label=metric.upper())
    ax.axhline(
        mean_value, color="tab:red", linestyle="--", linewidth=1.5,
        label=f"Generated-frame mean: {mean_value:.4g}"
    )
    ax.set_ylabel(metric.upper())
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

axes[-1].set_xlabel("Generated rollout frame")
plt.show()
